# * Amazon Deals Scraper

Scraper for **Amazon Today's Deals**.

### Highlights:
- **Async Playwright API**: Uses Playwright's native `async_api` to run seamlessly inside Google Colab's asyncio kernel without loop collisions.
- **Anti-Bot Defense Evasion**: Injects stealth scripts to mask `navigator.webdriver = undefined`, uses modern desktop Chrome user-agents, and mimics real browser headers.
- **Human-like Scrolling**: Smoothly scrolls in random increments (500–850px) with humanized pauses (1.2–2.2s) to dynamically trigger lazy loading of deals.
- **High-Speed BeautifulSoup Extraction**: Obtains the fully-rendered DOM source via `await page.content()` and parses it rapidly with BeautifulSoup.
- **Multi-Page Pagination**: Automatically navigates through event sale pages (`/events/sale/2/`, `/3/`, etc.) to hit your target goal (default: 300 deals).
- **Pandas & CSV Export**: Automatically organizes extracted fields into a Pandas DataFrame and downloads the CSV with one click.

## * Install Dependencies & Chromium (with System Libraries)
Python packages and Chromium along with system libraries (`--with-deps`).

In [4]:
# Install Python packages
!pip install -q playwright beautifulsoup4 pandas lxml

# Install Chromium and required Linux OS libraries (libatk, etc.)
!playwright install --with-deps chromium

Installing dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

## * Scraper Implementation (Async Playwright + BeautifulSoup)
This cell defines the core modular functions for stealth browser automation, humanized scrolling, and BeautifulSoup parsing.

In [7]:
import asyncio
import random
import re
import time
from urllib.parse import urljoin

from bs4 import BeautifulSoup
import pandas as pd
from playwright.async_api import async_playwright

# Realistic Desktop Chrome User-Agent
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/128.0.0.0 Safari/537.36"
)

# JavaScript injected before page scripts run to hide automation flags
STEALTH_SCRIPT = """
Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
window.chrome = { runtime: {} };
Object.defineProperty(navigator, 'languages', { get: () => ['en-US', 'en'] });
Object.defineProperty(navigator, 'plugins', { get: () => [1, 2, 3, 4, 5] });
"""

async def create_stealth_browser(playwright_instance, headless=True):
    """Initializes Chromium with anti-bot overrides and realistic context headers."""
    try:
        browser = await playwright_instance.chromium.launch(
            headless=headless,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--disable-infobars",
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                "--window-size=1920,1080"
            ]
        )
    except Exception as e:
        err_str = str(e)
        if "shared libraries" in err_str or "TargetClosedError" in type(e).__name__:
            print("\n" + "!" * 75)
            print("ERROR: Missing Linux libraries. Please run in a cell:")
            print("    !playwright install --with-deps chromium")
            print("!" * 75 + "\n")
        raise e

    context = await browser.new_context(
        user_agent=USER_AGENT,
        viewport={"width": 1920, "height": 1080},
        locale="en-US",
        timezone_id="America/New_York",
        extra_http_headers={"Accept-Language": "en-US,en;q=0.9"}
    )
    page = await context.new_page()
    await page.add_init_script(STEALTH_SCRIPT)
    return browser, context, page

async def check_anti_bot_detection(page) -> bool:
    """Detects if Amazon presented a CAPTCHA or Robot Check challenge."""
    try:
        title = (await page.title()).lower()
        if "robot check" in title or "captcha" in title:
            return True
        if await page.locator("form[action*='validateCaptcha']").count() > 0:
            return True
        body_text = await page.locator("body").inner_text()
        if "Enter the characters you see below" in body_text:
            return True
    except Exception:
        pass
    return False

def parse_deal_card(card, base_url="https://www.amazon.com") -> dict:
    """
    Extracts structured fields from an individual card with BeautifulSoup.
    Returns None if the card is not a real product (e.g. promo banners without ASIN).
    """
    # 1. Product Link & ASIN Extraction
    asin = card.get("data-asin", "").strip()
    link_el = (
        card.find("a", class_=re.compile(r"dcl-product-link|a-link-normal", re.I))
        or card.find("a", href=re.compile(r"/dp/|/deal/"))
        or card.find("a", href=True)
    )

    product_url = ""
    if link_el and link_el.get("href"):
        product_url = urljoin(base_url, link_el["href"])
        if not asin:
            asin_match = re.search(r"/dp/([A-Z0-9]{10})", product_url)
            if asin_match:
                asin = asin_match.group(1)

    # STRICT FILTER: Discard non-products (e.g. 'Restock on basics under $15' promo banners)
    # A valid Amazon product deal MUST have a 10-character alphanumeric ASIN
    if not asin or len(asin) != 10 or not asin.isalnum():
        return None

    # Canonical clean product link
    product_url = f"https://www.amazon.com/dp/{asin}"

    # 2. Product Title
    title = ""
    title_el = (
        card.find("span", class_=re.compile(r"dcl-product-label|deal-title|title|a-size-base-plus", re.I))
        or card.find("h2")
        or card.find("a", attrs={"data-testid": "deal-card-title"})
        or card.find("a", class_=re.compile(r"title", re.I))
    )
    if title_el:
        title = title_el.get_text(" ", strip=True)
    if not title:
        img = card.find("img")
        if img and img.get("alt"):
            title = img["alt"].strip()

    if not title or len(title) < 3:
        return None

    # 3. Deal Price
    deal_price = ""
    offscreen_price = card.find("span", class_="a-offscreen")
    if offscreen_price:
        deal_price = offscreen_price.get_text(strip=True)
    if not deal_price:
        p_el = card.find("span", class_=re.compile(r"price|a-color-price", re.I))
        deal_price = p_el.get_text(strip=True) if p_el else ""

    # 4. List Price (original / pre-discount)
    strike_el = (
        card.find("span", class_=re.compile(r"dcl-product-price-old|a-text-price|a-text-strike"))
        or card.find("div", class_=re.compile(r"old-price"))
    )
    list_price = ""
    if strike_el:
        off = strike_el.find("span", class_="a-offscreen")
        list_price = off.get_text(strip=True) if off else strike_el.get_text(strip=True)

    # 5. Discount Badges
    badge = (
        card.find("div", class_=re.compile(r"_badgeContainer_|dcl-badge", re.I))
        or card.find("span", class_=re.compile(r"deal-badge|badge-text|percentage", re.I))
        or card.find("span", class_="a-badge-text")
    )
    discount = badge.get_text(" ", strip=True) if badge else ""

    # 6. Customer Rating & Reviews
    rating_el = card.find("span", class_="a-icon-alt") or card.find("i", class_=re.compile(r"a-icon-star", re.I))
    rating = rating_el.get_text(strip=True) if rating_el else ""

    rev_el = (
        card.find("span", attrs={"aria-label": re.compile(r"ratings|reviews", re.I)})
        or card.find("a", href=re.compile(r"#customerReviews", re.I))
    )
    reviews_count = rev_el.get_text(strip=True) if rev_el else ""

    # 7. Thumbnail Image
    img_tag = card.find("img")
    image_url = img_tag.get("src", "") if img_tag else ""

    return {
        "asin": asin,
        "title": title,
        "deal_price": deal_price,
        "list_price": list_price,
        "discount": discount,
        "rating": rating,
        "reviews_count": reviews_count,
        "product_url": product_url,
        "image_url": image_url
    }

def extract_deals_from_html(html_source: str, seen_asins: set, target_max: int) -> list:
    """Parses page HTML with BeautifulSoup and extracts new valid product deals."""
    soup = BeautifulSoup(html_source, "html.parser")

    card_selectors = [
        "div.dcl-product", "li.a-carousel-card",
        "div[data-testid='deal-card']",
        "div[data-component-type='s-search-result']",
        "div[data-asin]:not([data-asin=''])"
    ]

    elements = []
    for sel in card_selectors:
        found = soup.select(sel)
        if len(found) > len(elements):
            elements = found

    new_items = []
    for card in elements:
        deal = parse_deal_card(card)
        if not deal:
            continue

        asin = deal["asin"]
        if asin in seen_asins:
            continue

        seen_asins.add(asin)
        new_items.append(deal)
        if len(seen_asins) >= target_max:
            break

    return new_items

async def scrape_amazon_deals(url="https://www.amazon.com/events/labordaysale", max_deals=300, output_csv="amazon_deals.csv") -> pd.DataFrame:
    """
    Scrapes Amazon Today's Deals by scrolling smoothly and clicking 'View more deals'.
    If the deals feed stalls or 'View more deals' ceases to load new items,
    it automatically clears cookies and performs a full page refresh to fetch a fresh deals session.
    """
    print("=" * 75)
    print("AMAZON TODAY'S DEALS SCRAPER")
    print(f"Target URL:  {url}")
    print(f"Goal:        Collect {max_deals} actual product deals")
    print("=" * 75)

    all_deals = []
    seen_asins = set()
    consecutive_no_new_deals = 0
    button_clicks = 0
    refresh_count = 0

    async with async_playwright() as p:
        browser, context, page = await create_stealth_browser(p, headless=True)
        try:
            print(f"[*] Navigating to {url}...")
            await page.goto(url, wait_until="domcontentloaded", timeout=60000)

            if await check_anti_bot_detection(page):
                print("    [!] Bot challenge detected. Pausing 5s...")
                await asyncio.sleep(5)
            else:
                print("    [+] Anti-bot check passed successfully.")

            await asyncio.sleep(random.uniform(2.5, 3.5))

            view_more_selector = (
                "button[data-testid='load-more-view-more-button'], "
                "button.ButtonLink-module__root_eh2cgp8M2THLjamsgeRE, "
                "[data-testid='load-more-view-more-button'], "
                "button:has-text('View more deals')"
            )

            # Continuous collection loop until max_deals is reached
            while len(all_deals) < max_deals:
                # 1. Smooth human scrolling in 3 increments
                for _ in range(3):
                    scroll_step = random.randint(650, 950)
                    await page.evaluate(f"window.scrollBy({{ top: {scroll_step}, behavior: 'smooth' }});")
                    await asyncio.sleep(random.uniform(1.0, 1.8))

                # 2. Check for and click the 'View more deals' button
                view_more_btn = page.locator(view_more_selector)
                btn_count = await view_more_btn.count()
                if btn_count > 0 and await view_more_btn.first.is_visible():
                    button_clicks += 1
                    print(f"    [Click #{button_clicks}] Clicking 'View more deals' button to load more products...")
                    await view_more_btn.first.scroll_into_view_if_needed()
                    await asyncio.sleep(random.uniform(0.5, 1.0))
                    await view_more_btn.first.click()
                    await asyncio.sleep(random.uniform(2.5, 3.5))

                # 3. Extract page DOM via page.content() and parse with BeautifulSoup
                html_source = await page.content()
                new_deals = extract_deals_from_html(html_source, seen_asins=seen_asins, target_max=max_deals)

                if len(new_deals) > 0:
                    all_deals.extend(new_deals)
                    consecutive_no_new_deals = 0
                    print(f"    [+] Loaded {len(new_deals)} new deals -> Total collected: {len(all_deals)}/{max_deals}")
                else:
                    consecutive_no_new_deals += 1

                if len(all_deals) >= max_deals:
                    print(f"\n[✓] Successfully collected target of {len(all_deals)} valid product deals!")
                    break

                # 4. If stalled, try a nudge scroll to trigger dynamic lazy loading
                if consecutive_no_new_deals == 3:
                    print("    [!] Nudging scroll position to trigger lazy loading...")
                    await page.evaluate("window.scrollBy(0, -600);")
                    await asyncio.sleep(1.0)
                    await page.evaluate("window.scrollBy(0, 1200);")
                    await asyncio.sleep(2.0)

                # 5. AUTO-REFRESH & COOKIE RESET:
                # If no new deals after 5 consecutive attempts (nudge + button), perform a clean session refresh
                if consecutive_no_new_deals >= 5:
                    refresh_count += 1
                    print(f"\n    [🔄 Refresh #{refresh_count}] Progress stalled at {len(all_deals)}/{max_deals}.")
                    print("    [🔄] Clearing cookies and refreshing page to start a fresh deals session...")

                    # Close page, clear cookies and storage to reset the session completely
                    await page.close()
                    await context.clear_cookies()
                    page = await context.new_page()
                    await page.add_init_script(STEALTH_SCRIPT)

                    # Re-navigate to deals page with fresh session
                    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                    await asyncio.sleep(random.uniform(3.5, 4.5))
                    consecutive_no_new_deals = 0
                    button_clicks = 0

        finally:
            print("\n[*] Closing browser session...")
            await context.close()
            await browser.close()

    df = pd.DataFrame(all_deals)
    if output_csv and not df.empty:
        df.to_csv(output_csv, index=False, encoding="utf-8")
        print(f"[✓] Saved {len(df)} actual product deals to '{output_csv}'")
    return df

## * Scraper Run
Scrapes up to 300 deals.

Note: Google Colab natively supports top-level `await`.

In [8]:
# Target Amazon deals URL and target deal count
TARGET_URL = "https://www.amazon.com/events/labordaysale"  # or https://www.amazon.com/deals
MAX_DEALS = 300
OUTPUT_CSV = "amazon_deals.csv"

# Run the scraper directly with top-level await
deals_df = await scrape_amazon_deals(url=TARGET_URL, max_deals=MAX_DEALS, output_csv=OUTPUT_CSV)

AMAZON TODAY'S DEALS SCRAPER
Target URL:  https://www.amazon.com/events/labordaysale
Goal:        Collect 300 actual product deals
[*] Navigating to https://www.amazon.com/events/labordaysale...
    [+] Anti-bot check passed successfully.
    [Click #1] Clicking 'View more deals' button to load more products...
    [+] Loaded 27 new deals -> Total collected: 27/300
    [Click #2] Clicking 'View more deals' button to load more products...
    [+] Loaded 33 new deals -> Total collected: 60/300
    [Click #3] Clicking 'View more deals' button to load more products...
    [+] Loaded 44 new deals -> Total collected: 104/300
    [Click #4] Clicking 'View more deals' button to load more products...
    [+] Loaded 9 new deals -> Total collected: 113/300
    [Click #5] Clicking 'View more deals' button to load more products...
    [Click #6] Clicking 'View more deals' button to load more products...
    [Click #7] Clicking 'View more deals' button to load more products...
    [!] Nudging scroll

## * Preview & Inspect the Results
Inspect the extracted deals table and summary metrics.

In [9]:
print(f"Total Deals Retrieved: {len(deals_df)}")
print(f"Rows with non-empty ASIN: {deals_df['asin'].notna().sum()}/{len(deals_df)}\n")

# Display top 10 deals
cols = [c for c in ["asin", "title", "deal_price", "discount", "rating", "product_url"] if c in deals_df.columns]
deals_df[cols].head(10)

Total Deals Retrieved: 300
Rows with non-empty ASIN: 300/300



,asin,title,deal_price,discount,rating,product_url
0,B09XS7JWHH,Sony WH-1000XM5 Premium Noise Cancelling Wirel...,"Deal Price: TWD 6,259.67",50% off Ends in 08:20:06,,https://www.amazon.com/dp/B09XS7JWHH
1,B0BDCT78KQ,GCI Outdoor Freestyle Rocker XL Camping Chair ...,"Deal Price: TWD 2,149.79",20% off Limited time deal,,https://www.amazon.com/dp/B0BDCT78KQ
2,B0FH655984,SUPFINE Magnetic for iPhone 17 Pro Max Case Black,Deal Price: TWD 176.41,44% off Ends in 01:25:07,,https://www.amazon.com/dp/B0FH655984
3,B0B2RM68G2,"BIODANCE Bio-Collagen Real Deep Mask, Hydratin...",Deal Price: TWD 510.57,15% off Limited time deal,,https://www.amazon.com/dp/B0B2RM68G2
4,B0D1CXL52G,"Ninja Flip Toaster Oven & Air Fryer, 8-in-1, S...","Deal Price: TWD 4,740.28",40% off Limited time deal,,https://www.amazon.com/dp/B0D1CXL52G
5,B09MR4B13C,"NapQueen 8 Inch Twin Size Mattress, Bamboo Cha...","Deal Price: TWD 2,557.93",16% off Limited time deal,,https://www.amazon.com/dp/B09MR4B13C
6,B07HC4X8DJ,PlushDeluxe Bamboo Waterproof Mattress Protect...,Deal Price: TWD 948.12,17% off Limited time deal,,https://www.amazon.com/dp/B07HC4X8DJ
7,B0C1T9M4PQ,"Hanes Comfortblend EcoSmart Hoodie, Midweight ...",Deal Price: TWD 252.92,70% off Limited time deal,,https://www.amazon.com/dp/B0C1T9M4PQ
8,B01GNVF7YI,Waterpik Cordless Advanced Portable Water Flos...,"Deal Price: TWD 1,896.24",40% off Limited time deal,,https://www.amazon.com/dp/B01GNVF7YI
9,B0F6LM95F2,"YaFex Curtain Rods 32-144 Inch, Heavy Duty 1 I...",Deal Price: TWD 663.59,16% off Ends in 08:20:06,,https://www.amazon.com/dp/B0F6LM95F2


## * Creation of CSV File
Run this cell in Google Colab to download `amazon_deals.csv` directly to your local machine.

In [10]:
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
    print(f"[✓] Download triggered for {OUTPUT_CSV}")
except ImportError:
    print(f"Not running inside Google Colab. The CSV is saved locally at: {OUTPUT_CSV}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[✓] Download triggered for amazon_deals.csv
